# Multi-scale Coarse-to-Fine Registration of Left/Right Mammograms

**Goal:** Register a left mammogram to a right one (or vice versa) after horizontally mirroring one of them, so that we can later compare regional texture between the two breasts.

**Method (didactic version):**
1. Mirror one image so both breasts point the same way.
2. Build a Gaussian image pyramid (factor 2 per level, ~5 levels → ~32× downsampling at the coarsest level).
3. Estimate a transform at the coarsest level using gradient-based optimization of mean squared error.
4. Refine the estimate level by level, using the previous result as the starting point.
5. Choose between **rigid** (translation + rotation) and **affine** (6 parameters) via a single switch.

We use only `numpy`, `scipy`, and `matplotlib` to keep the code transparent. No specialized registration library is hidden behind the scenes — every step is visible.

**Note on metric:** MSE works here because both images come from the same modality and similar acquisition. For cross-modality you would replace it with mutual information.

## 1. Imports and configuration

In [2]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter, affine_transform, zoom
from scipy.optimize import minimize
from pathlib import Path

# --- User settings -----------------------------------------------------------
MODE = 'rigid'          # 'rigid' (3 params) or 'affine' (6 params)
PIXEL_SIZE_MM = 0.1     # used only for reporting results in mm, TODO: load resolution from DICOM tags 
N_LEVELS = 6            # 6 levels => coarsest is downsampled by 2^5 = 32
MIRROR_MOVING = True    # mirror the moving image left-right before registration

# loading two datasets from /kaggle/input/datasets/kmader/mias-mammography/
# TODO: file selection to easily try various cases
FIXED_PATH  = '/kaggle/input/datasets/kmader/mias-mammography/all-mias/mdb015.pgm'  # reference image, right breast (odd numbered images)
MOVING_PATH = '/kaggle/input/datasets/kmader/mias-mammography/all-mias/mdb016.pgm'   # will be mirrored & warped, left breast (even numbered images)

rng = np.random.default_rng(0)

## 2. Loading images

We accept PNG/JPG via `matplotlib.image.imread` and DICOM via `pydicom` if available. The image is converted to a float array in [0, 1]. For mammograms, intensity normalization helps the optimizer behave consistently across pyramid levels.

In [ ]:
def load_image(path):
    """Load a mammogram as a 2D float array, normalized to [0, 1]."""
    path = str(path)
    if path.lower().endswith(('.dcm', '.dicom')):
        import pydicom
        img = pydicom.dcmread(path).pixel_array.astype(np.float32)
    else:
        import matplotlib.image as mpimg
        img = mpimg.imread(path).astype(np.float32)
        if img.ndim == 3:                  # drop alpha / convert RGB to gray
            img = img[..., :3].mean(axis=-1)
    img -= img.min()
    if img.max() > 0:
        img /= img.max()
    return img

# For a quick offline test you can uncomment the next two lines to fabricate data:
# fixed  = np.zeros((3000, 2000), np.float32); fixed[800:2400, 400:1600] = np.linspace(0,1,1600)[None,:]
# moving = np.roll(np.fliplr(fixed), shift=(40, -25), axis=(0, 1))

fixed  = load_image(FIXED_PATH)
moving = load_image(MOVING_PATH)

if MIRROR_MOVING:
    moving = np.ascontiguousarray(moving[:, ::-1])

print(f'fixed:  {fixed.shape}, dtype={fixed.dtype}')
print(f'moving: {moving.shape}, dtype={moving.dtype}  (mirrored={MIRROR_MOVING})')

## 3. Image pyramid

A Gaussian pyramid removes high-frequency content before downsampling, which prevents aliasing and — more importantly for registration — gives the optimizer a smooth, low-dimensional cost surface at coarse scales. We apply a Gaussian with σ ≈ 1.0 (in the downsampled pixel grid) before each 2× decimation.

In [ ]:
def build_pyramid(img, n_levels):
    """Return a list [level_0, level_1, ...] where level_0 is full resolution."""
    pyr = [img]
    for _ in range(n_levels - 1):
        smoothed = gaussian_filter(pyr[-1], sigma=1.0)
        pyr.append(smoothed[::2, ::2])
    return pyr

fixed_pyr  = build_pyramid(fixed,  N_LEVELS)
moving_pyr = build_pyramid(moving, N_LEVELS)

fig, axes = plt.subplots(2, N_LEVELS, figsize=(2.2 * N_LEVELS, 5))
for k in range(N_LEVELS):
    axes[0, k].imshow(fixed_pyr[k],  cmap='gray'); axes[0, k].set_title(f'fixed L{k}\n{fixed_pyr[k].shape}')
    axes[1, k].imshow(moving_pyr[k], cmap='gray'); axes[1, k].set_title(f'moving L{k}\n{moving_pyr[k].shape}')
    for ax in (axes[0, k], axes[1, k]): ax.axis('off')
plt.tight_layout(); plt.show()

## 4. Parameterizing the transform

We parameterize transforms around the **image center** so that small rotations don't cause large translations as a side effect — this makes optimization much easier.

- **rigid (3 params):** `[tx, ty, theta]` → translation in pixels and rotation in radians.
- **affine (6 params):** `[a11, a12, a21, a22, tx, ty]` parameterized as identity + small perturbation, so the zero vector means "no transform".

`scipy.ndimage.affine_transform` expects the inverse mapping: `input_coord = M @ output_coord + offset`. We build that explicitly.

In [ ]:
def params_to_matrix(params, mode, center):
    """Convert parameter vector to (M, offset) for affine_transform centered at `center`.

    The forward transform is:  x_moving = M (x_fixed - c) + c + t
    which is what affine_transform consumes when called with `matrix=M, offset=c + t - M@c`.
    """
    cy, cx = center
    if mode == 'rigid':
        tx, ty, theta = params
        M = np.array([[ np.cos(theta), -np.sin(theta)],
                      [ np.sin(theta),  np.cos(theta)]])
    elif mode == 'affine':
        a11, a12, a21, a22, tx, ty = params
        M = np.eye(2) + np.array([[a11, a12], [a21, a22]])
    else:
        raise ValueError(f'unknown mode {mode!r}')
    c = np.array([cy, cx])
    t = np.array([ty, tx])
    offset = c + t - M @ c
    return M, offset

def warp(moving_img, params, mode):
    """Warp `moving_img` into the fixed image's frame."""
    center = (np.array(moving_img.shape) - 1) / 2.0
    M, offset = params_to_matrix(params, mode, center)
    return affine_transform(moving_img, M, offset=offset, order=1,
                            mode='constant', cval=0.0, prefilter=False)

def initial_params(mode):
    return np.zeros(3 if mode == 'rigid' else 6, dtype=np.float64)

## 5. Scaling parameters between pyramid levels

When we go from a coarse level to a finer one, **rotations and the affine matrix part stay the same** (they are dimensionless), but **translations double** because they are expressed in pixels and the pixel grid just doubled in resolution.

In [ ]:
def upscale_params(params, mode, factor=2.0):
    p = params.copy()
    if mode == 'rigid':
        p[0] *= factor   # tx
        p[1] *= factor   # ty
    else:                # affine: last two entries are tx, ty
        p[4] *= factor
        p[5] *= factor
    return p

## 6. Cost function and optimization at one level

We minimize mean squared error inside a mask of pixels where the warped moving image is valid (i.e. not the zero background introduced by `affine_transform`). This avoids the optimizer cheating by translating the breast out of the frame.

In [ ]:
def mse_cost(params, fixed_img, moving_img, mode):
    warped = warp(moving_img, params, mode)
    mask = warped > 0                         # exclude out-of-frame pixels
    if mask.sum() < 0.1 * mask.size:          # degenerate transform
        return 1.0
    diff = (warped - fixed_img)[mask]
    return float(np.mean(diff * diff))

def register_at_level(fixed_img, moving_img, init_params, mode, maxiter=60):
    # Powell is derivative-free and robust for low-dimensional registration.
    res = minimize(mse_cost, init_params, args=(fixed_img, moving_img, mode),
                   method='Powell',
                   options={'xtol': 1e-3, 'ftol': 1e-5, 'maxiter': maxiter})
    return res.x, res.fun

## 7. The coarse-to-fine driver

Walk the pyramid from coarsest to finest. At each level: optimize, then upscale the parameters before handing them to the next level.

In [ ]:
def multiscale_register(fixed_pyr, moving_pyr, mode):
    params = initial_params(mode)
    history = []
    # Iterate from the coarsest level (last in the list) down to level 0.
    for level in range(len(fixed_pyr) - 1, -1, -1):
        f, m = fixed_pyr[level], moving_pyr[level]
        params, cost = register_at_level(f, m, params, mode)
        history.append((level, cost, params.copy()))
        print(f'level {level:>2}  shape={f.shape}  cost={cost:.6f}  params={np.round(params, 4)}')
        if level > 0:
            params = upscale_params(params, mode, factor=2.0)
    return params, history

final_params, history = multiscale_register(fixed_pyr, moving_pyr, MODE)

## 8. Apply the final transform and inspect the result

We show the fixed image, the original moving image, the warped moving image, and a difference map. A good registration leaves only soft, low-contrast structure in the difference.

In [ ]:
warped = warp(moving, final_params, MODE)
diff   = warped - fixed

fig, ax = plt.subplots(1, 4, figsize=(16, 6))
ax[0].imshow(fixed,  cmap='gray');  ax[0].set_title('fixed')
ax[1].imshow(moving, cmap='gray');  ax[1].set_title('moving (mirrored)')
ax[2].imshow(warped, cmap='gray');  ax[2].set_title('moving → fixed')
im = ax[3].imshow(diff, cmap='seismic', vmin=-0.5, vmax=0.5)
ax[3].set_title('difference (warped − fixed)')
for a in ax: a.axis('off')
plt.colorbar(im, ax=ax[3], fraction=0.04); plt.tight_layout(); plt.show()

if MODE == 'rigid':
    tx, ty, theta = final_params
    print(f'translation:  ({tx*PIXEL_SIZE_MM:+.2f}, {ty*PIXEL_SIZE_MM:+.2f}) mm')
    print(f'rotation:     {np.degrees(theta):+.3f} deg')
else:
    a11, a12, a21, a22, tx, ty = final_params
    A = np.eye(2) + np.array([[a11, a12], [a21, a22]])
    print(f'linear part A =\n{A}')
    print(f'translation:  ({tx*PIXEL_SIZE_MM:+.2f}, {ty*PIXEL_SIZE_MM:+.2f}) mm')

## 9. Convergence check

The cost should drop monotonically (or close to it) as we move from coarse to fine. If it suddenly goes up at the finest level, that usually means the optimizer fell into a local minimum at a coarse level and couldn't recover — try more pyramid levels or a smaller initial step.

In [ ]:
levels = [h[0] for h in history]
costs  = [h[1] for h in history]
plt.figure(figsize=(6, 3.5))
plt.plot(range(len(levels)), costs, 'o-')
plt.xticks(range(len(levels)), [f'L{l}' for l in levels])
plt.xlabel('pyramid level (coarse → fine)'); plt.ylabel('MSE at convergence')
plt.title('Cost across pyramid levels'); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

## 10. Where to go from here

- **Better metric:** mutual information for inter-modality, or normalized cross-correlation if intensity scaling differs between left and right.
- **Masking the breast:** estimate a breast mask (e.g. Otsu threshold + largest connected component) and evaluate the cost only inside it. This avoids the air background dominating.
- **Non-rigid step:** after this rigid/affine alignment, add a B-spline or demons step for the local deformation that actually differs between the two breasts. That's typically what you want before regional texture comparison.
- **Robustness:** wrap the whole pipeline in a few random restarts and keep the lowest final MSE.